# CHECKPOINT 4.1 — QUẢN LÝ HỌC SINH + PHỤ HUYNH (STUDENT & PARENT MANAGEMENT)

Notebook này minh họa cấu trúc dữ liệu, quy tắc nghiệp vụ THCS, cơ chế liên kết 2 chiều (Bidirectional Linking), nguyên tắc Soft Delete, và quy trình phân quyền RBAC của Phân hệ Quản lý Học sinh & Phụ huynh trong hệ thống **SmartEdu**.

## 1. QUY MÔ VÀ GIỚI HẠN DỮ LIỆU THCS
- **Khối lớp:** 6, 7, 8, 9 (Tuyệt đối không có Tiểu học hay THPT)
- **Số lượng học sinh:** 216 học sinh (18 học sinh / lớp, 12 lớp)
- **Mã định danh nghiệp vụ:**
  - Học sinh: `STU-2026-XXX`
  - Phụ huynh: `PAR-2026-XXX`

In [1]:
valid_grades = [6, 7, 8, 9]
def validate_grade(grade):
    return grade in valid_grades

student_id = f"STU-2026-{1:03d}"
parent_id = f"PAR-2026-{1:03d}"

print(f"✓ Khối lớp hợp lệ: {valid_grades}")
print(f"✓ Mẫu Mã Học sinh: {student_id}")
print(f"✓ Mẫu Mã Phụ huynh: {parent_id}")

✓ Khối lớp hợp lệ: [6, 7, 8, 9]
✓ Mẫu Mã Học sinh: STU-2026-001
✓ Mẫu Mã Phụ huynh: PAR-2026-001


## 2. LIÊN KẾT HAi CHIỀU (BIDIRECTIONAL LINKING: STUDENT ↔ PARENT)
Mối quan hệ hai chiều được đồng bộ qua Firestore `writeBatch`:
- `students.parentId` & `students.parentIds` ↔ `parents.id`
- `parents.studentIds` & `parents.childIds` ↔ `students.id`

In [2]:
student = {
    "id": "STU-2026-001",
    "name": "Nguyễn Minh Anh",
    "grade": 6,
    "parentId": "PAR-2026-001",
    "parentIds": ["PAR-2026-001"],
    "parentName": "Nguyễn Văn Hùng",
    "status": "ACTIVE"
}

parent = {
    "id": "PAR-2026-001",
    "name": "Nguyễn Văn Hùng",
    "studentIds": ["STU-2026-001"],
    "childIds": ["STU-2026-001"],
    "status": "ACTIVE"
}

print(f"Học sinh: {student['name']} (ID: {student['id']}) | Phụ huynh ID: {student['parentId']}")
print(f"Phụ huynh: {parent['name']} (ID: {parent['id']}) | Danh sách con: {parent['childIds']}")
assert parent['id'] in student['parentIds']
assert student['id'] in parent['studentIds']
print("✓ Đồng bộ liên kết 2 chiều thành công!")

Học sinh: Nguyễn Minh Anh (ID: STU-2026-001) | Phụ huynh ID: PAR-2026-001
Phụ huynh: Nguyễn Văn Hùng (ID: PAR-2026-001) | Danh sách con: ['STU-2026-001']
✓ Đồng bộ liên kết 2 chiều thành công!


## 3. PHÂN QUYỀN RBAC (ROLE-BASED ACCESS CONTROL)
- **Quyền Đăng Ký / Chỉnh Sửa / Xóa Mềm:** `ADMIN`, `OWNER`, `ACADEMIC_STAFF`
- **Quyền Xem Chi Tiết:** `ADMIN`, `OWNER`, `ACADEMIC_STAFF`, `ACCOUNTANT`, `TEACHER`
- **Quyền Xem Giới Hạn:** `STUDENT` (chỉ xem bản thân) và `PARENT` (chỉ xem con cái thuộc `childIds`)

In [3]:
def can_write_student_parent(role):
    return role in ["ADMIN", "OWNER", "ACADEMIC_STAFF"]

for r in ["ADMIN", "ACADEMIC_STAFF", "TEACHER", "STUDENT", "PARENT"]:
    print(f"Role {r} write access: {can_write_student_parent(r)}")

Role ADMIN write access: True
Role ACADEMIC_STAFF write access: True
Role TEACHER write access: False
Role STUDENT write access: False
Role PARENT write access: False
